# 🌸 Classificação Botânica com Deep Learning (Oxford Flowers-102)
## Protocolo Experimental Reproduzível para Google Colab Pro (GPU)

Este notebook executa de ponta a ponta o pipeline auditado para categorização botânica de grão fino (*Fine-Grained Visual Categorization — FGVC*) utilizando **MobileNetV2** no benchmark oficial **Oxford Flowers-102** ($8.189$ imagens em $102$ classes: $1.020$ treino / $1.020$ validação / $6.149$ teste).

### 📌 Etapas de Reprodutibilidade e Rastreabilidade:
1. **Diagnóstico Completo de Hardware e Software:** Registra a GPU real alocada na sessão (T4, L4, A100, etc.), CPU, RAM, CUDA, cuDNN, PyTorch, Torchvision e demais dependências.
2. **Montagem Opcional do Google Drive:** Garante que os artefatos binários (`metrics.json`, `predictions.npz`, `checkpoint_best_p1.pth`, `checkpoint_best_ft.pth` e `plots/*.png`) sejam salvos fora do armazenamento efêmero `/content`.
3. **Clonagem, Commit Hash e Dependências:** Registra o commit exato (`git rev-parse HEAD`) que gerou os resultados.
4. **Validação Prévia de Software:** Executa os $16$ testes unitários (`pytest test_pipeline.py -v`) e o `--smoke-test` sintético.
5. **Treinamento e Avaliação Completa (`pipeline_botanico.py`):** Extração determinística de embeddings, modelos clássicos, Fase 1 (`backbone.eval()`), Fase 2 (Fine-Tuning `features.16-18`), TTA (média imagem original + reflexão horizontal nas $6.149$ imagens), McNemar Edwards, Bootstrap pareado, calibração (ECE, Brier, NLL) e Grad-CAM.
6. **Auditoria Independente a partir de `predictions.npz`:** Recalcula todas as métricas diretamente das probabilidades salvas (`--verify-only`).

### 1. Diagnóstico Real de Hardware, Sistema e Versões
Não presuma que a GPU alocada pelo Colab Pro será sempre a mesma. Esta célula registra a GPU real da sessão, memória VRAM, CPU, RAM e versões exatas das bibliotecas.

In [ ]:
!nvidia-smi
!lscpu | grep -E "Model name|Socket|Thread|NUMA|CPU\(s\)" || true
!free -h

import sys, platform, torch, torchvision, numpy, scipy, sklearn, pandas, matplotlib
print("=" * 70)
print(f"Sistema Operacional : {platform.platform()}")
print(f"Python              : {platform.python_version()} ({sys.version.split()[0]})")
print(f"PyTorch             : {torch.__version__}")
print(f"Torchvision         : {torchvision.__version__}")
print(f"CUDA Disponível     : {torch.cuda.is_available()} (Versão CUDA: {torch.version.cuda})")
print(f"cuDNN Versão        : {torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else 'N/A'}")
if torch.cuda.is_available():
    print(f"GPU Alocada         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"NumPy / SciPy       : {numpy.__version__} / {scipy.__version__}")
print(f"Scikit-Learn        : {sklearn.__version__}")
print("=" * 70)

### 2. Montagem do Google Drive para Persistência Permanente dos Artefatos
Evita a perda de `metrics.json`, `predictions.npz` e `checkpoint_best_ft.pth` caso a máquina virtual do Colab seja reciclada.

In [ ]:
USE_GOOGLE_DRIVE = True
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/classificacao_botanica_artefatos"

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print(f"[OK] Google Drive montado. Backup configurado em: {DRIVE_BACKUP_DIR}")
    except Exception as e:
        print(f"[AVISO] Execução fora do Google Colab ou Drive não montado: {e}")
        DRIVE_BACKUP_DIR = None
else:
    DRIVE_BACKUP_DIR = None

### 3. Clonagem do Repositório, Registro do Commit e Instalação de Dependências

In [ ]:
%cd /content
import os
if os.path.exists("/content/classificacao-botanica"):
    !rm -rf /content/classificacao-botanica

!git clone https://github.com/kelryana/classificacao-botanica.git
%cd /content/classificacao-botanica
!git log -n 1 --oneline
!pip install -q -r requirements.txt

### 4. Validação da Suíte de Testes Unitários (16 Testes) e Smoke Test Sintético
Verifica a integridade das métricas, calibração ECE (incluindo probabilidade 1,0 e bins vazios), McNemar com correção de Edwards, Bootstrap pareado, extração determinística de embeddings, congelamento estrito de `running_mean`/`running_var` no `BatchNorm2d` e recálculo via `predictions.npz`.

In [ ]:
!pytest test_pipeline.py -v
!python pipeline_botanico.py --smoke-test

### 5. Execução Completa do Pipeline Científico (`pipeline_botanico.py`)
Parâmetros efetivos registrados no código:
- **Semente (`seed`):** `42` (`cudnn.deterministic = True`, `cudnn.benchmark = False`)
- **Fase 1:** `8` épocas, `AdamW(lr=1e-3, weight_decay=1e-4)`, `backbone.eval()`, seleção pelo menor `val_loss`
- **Fase 2 (Fine-Tuning `features.16-18`):** `4` épocas, `AdamW(lr_backbone=1e-4, lr_head=5e-4, weight_decay=1e-4)`, sem scheduler de decaimento, seleção pelo menor `val_loss`
- **UMAP:** ajustado sobre `train_val` (`--umap-split train_val --umap-max-samples 1500`) para evitar exploração adicional do conjunto de teste
- **TTA:** média aritmética simples das probabilidades Softmax entre imagem original e reflexão horizontal nas $6.149$ amostras de teste.

In [ ]:
drive_arg = f"--drive-backup-dir {DRIVE_BACKUP_DIR}" if DRIVE_BACKUP_DIR else ""
!python pipeline_botanico.py --base-dir ./resultados_botanico --data-dir ./data --epochs-p1 8 --epochs-p2 4 --batch-size 32 --umap-split train_val --umap-max-samples 1500 {drive_arg}

### 6. Recálculo Independente e Auditoria a partir de `predictions.npz`
Confirma que todas as métricas, contagens discordantes $(b, c)$ de McNemar, intervalos Bootstrap e métricas de calibração são integralmente reproduzíveis a partir dos vetores de predição salvos.

In [ ]:
!python pipeline_botanico.py --verify-only ./resultados_botanico/predictions.npz

### 7. Empacotamento ZIP dos Artefatos para Download Local
Gera `resultados_botanico_auditados.zip` contendo `metrics.json`, `predictions.npz`, `checkpoint_best_p1.pth`, `checkpoint_best_ft.pth` e as 11 figuras em `plots/`.

In [ ]:
!zip -r /content/resultados_botanico_auditados.zip ./resultados_botanico
ls -lh ./resultados_botanico /content/resultados_botanico_auditados.zip

### 8. Inspeção Visual das 11 Figuras Técnicas Geradas (300 DPI)

In [ ]:
from pathlib import Path
from IPython.display import Image, display

plots = sorted(list(Path("./resultados_botanico/plots").glob("*.png")))
print(f"Total de figuras geradas: {len(plots)}\n")

for p in plots:
    print(f"📊 Exibindo: {p.name}")
    display(Image(filename=str(p), width=800))
    print("-" * 60)